## 1. Download AgriFlow Raw Data
Download the five AgriFlow source files into the local mounted data folder so they can be loaded into HDFS in the next section.

In [1]:
import os
import urllib.request

LOCAL_DATA = "/home/jovyan/data/agriflow-raw"
os.makedirs(LOCAL_DATA, exist_ok=True)

FILES = {
    "crop-yields.csv.gz": "https://raw.githubusercontent.com/prof-tcsmith/6562S26-data/main/final-projects/03-agriflow-farming/crop-yields.csv.gz",
    "equipment-usage.json.gz": "https://raw.githubusercontent.com/prof-tcsmith/6562S26-data/main/final-projects/03-agriflow-farming/equipment-usage.json.gz",
    "market-prices.csv.gz": "https://raw.githubusercontent.com/prof-tcsmith/6562S26-data/main/final-projects/03-agriflow-farming/market-prices.csv.gz",
    "soil-sensors.json.gz": "https://raw.githubusercontent.com/prof-tcsmith/6562S26-data/main/final-projects/03-agriflow-farming/soil-sensors.json.gz",
    "weather-daily.csv.gz": "https://raw.githubusercontent.com/prof-tcsmith/6562S26-data/main/final-projects/03-agriflow-farming/weather-daily.csv.gz",
}

print("Local download folder:")
print(LOCAL_DATA)

Local download folder:
/home/jovyan/data/agriflow-raw


In [6]:
print("Downloading AgriFlow data...\n")

for filename, url in FILES.items():
    path = os.path.join(LOCAL_DATA, filename)

    if os.path.exists(path):
        print(f"[skip] {filename} already exists")
        continue

    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, path)
    print(f"[OK] {filename}")

print("\nDownload complete.")

HTTPError: HTTP Error 404: Not Found

In [2]:
print("Files now in local folder:\n")

local_files = sorted(os.listdir(LOCAL_DATA))
for f in local_files:
    print(" -", f)

Files now in local folder:

 - crop-yields.csv.gz
 - equipment-usage.json.gz
 - generate_data.py
 - market-prices.csv.gz
 - soil-sensors.json.gz
 - weather-daily.csv.gz


In [3]:
required_files = list(FILES.keys())

missing = []
for filename in required_files:
    full_path = os.path.join(LOCAL_DATA, filename)
    if not os.path.exists(full_path):
        missing.append(filename)

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("All required AgriFlow files are present.")

All required AgriFlow files are present.


## Step 2 Creating the Spark Session

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("agriflow-stage1-data-lake-setup")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("spark.pyspark.python", "/opt/conda/bin/python3")
    .config("spark.pyspark.driver.python", "/opt/conda/bin/python3")
    .getOrCreate()
)

print("Spark session created.")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

Spark session created.
Spark version: 3.5.0
Master: local[*]


## HDFS Structure and Dataset Mapping

In [5]:
BASE_HDFS = "hdfs://namenode:9000/agriflow"
LANDING = f"{BASE_HDFS}/landing"
CURATED = f"{BASE_HDFS}/curated"
ANALYTICS = f"{BASE_HDFS}/analytics"

DATASETS = {
    "crop-records": "crop-yields.csv.gz",
    "equipment-usage": "equipment-usage.json.gz",
    "market-prices": "market-prices.csv.gz",
    "soil-sensors": "soil-sensors.json.gz",
    "weather-data": "weather-daily.csv.gz",
}

print("Base HDFS path:", BASE_HDFS)
print("Landing zone:", LANDING)
print("Curated zone:", CURATED)
print("Analytics zone:", ANALYTICS)

Base HDFS path: hdfs://namenode:9000/agriflow
Landing zone: hdfs://namenode:9000/agriflow/landing
Curated zone: hdfs://namenode:9000/agriflow/curated
Analytics zone: hdfs://namenode:9000/agriflow/analytics


## Local File Check Before Load

In [6]:
import os

print("Checking local files before HDFS load...\n")

missing = []
for hdfs_name, filename in DATASETS.items():
    full_path = os.path.join(LOCAL_DATA, filename)
    if os.path.exists(full_path):
        print(f"[OK] {filename}")
    else:
        print(f"[MISSING] {filename}")
        missing.append(filename)

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("\nAll required files are ready for HDFS load.")

Checking local files before HDFS load...

[OK] crop-yields.csv.gz
[OK] equipment-usage.json.gz
[OK] market-prices.csv.gz
[OK] soil-sensors.json.gz
[OK] weather-daily.csv.gz

All required files are ready for HDFS load.


## Loading FIles into HDFS Landing Zone

In [8]:
loaded_counts = {}

for hdfs_name, filename in DATASETS.items():
    local_path = os.path.join(LOCAL_DATA, filename)
    hdfs_path = f"{LANDING}/{hdfs_name}"

    print(f"\nLoading {filename} -> {hdfs_path}")

    if filename.endswith(".json.gz"):
        df = (
            spark.read
            .option("multiLine", True)
            .json(f"file://{local_path}")
        )
    else:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(f"file://{local_path}")
        )

    row_count = df.count()

    if filename.endswith(".json.gz"):
        df.write.mode("overwrite").json(hdfs_path)
    else:
        df.write.mode("overwrite").option("header", True).csv(hdfs_path)

    loaded_counts[hdfs_name] = row_count
    print(f"[OK] {hdfs_name}: {row_count} rows written")


Loading crop-yields.csv.gz -> hdfs://namenode:9000/agriflow/landing/crop-records
[OK] crop-records: 100000 rows written

Loading equipment-usage.json.gz -> hdfs://namenode:9000/agriflow/landing/equipment-usage
[OK] equipment-usage: 80000 rows written

Loading market-prices.csv.gz -> hdfs://namenode:9000/agriflow/landing/market-prices
[OK] market-prices: 20000 rows written

Loading soil-sensors.json.gz -> hdfs://namenode:9000/agriflow/landing/soil-sensors
[OK] soil-sensors: 500000 rows written

Loading weather-daily.csv.gz -> hdfs://namenode:9000/agriflow/landing/weather-data
[OK] weather-data: 150000 rows written


## Verifying Landing Zone Contents

In [9]:
print("\n=== Landing Zone Verification ===")

for hdfs_name, filename in DATASETS.items():
    hdfs_path = f"{LANDING}/{hdfs_name}"

    try:
        if filename.endswith(".json.gz"):
            df = spark.read.json(hdfs_path)
        else:
            df = spark.read.option("header", True).csv(hdfs_path)

        count = df.count()
        print(f"{hdfs_path} -> {count} rows")
        df.show(3, truncate=False)

    except Exception as e:
        print(f"{hdfs_path} -> ERROR: {e}")


=== Landing Zone Verification ===
hdfs://namenode:9000/agriflow/landing/crop-records -> 100000 rows
+-------+-----------+----+---------+-----+----------------------+----------------------+------------------+-------------+------------+
|farm_id|field_id   |year|crop_type|acres|yield_bushels_per_acre|fertilizer_applied_lbs|irrigation_gallons|planting_date|harvest_date|
+-------+-----------+----+---------+-----+----------------------+----------------------+------------------+-------------+------------+
|FARM-01|FARM-01-F01|2017|corn     |134  |198.6                 |314.8                 |12733             |2017-04-11   |2017-10-20  |
|FARM-01|FARM-01-F01|2018|corn     |97   |194.1                 |166.1                 |44852             |2018-04-18   |2018-10-23  |
|FARM-01|FARM-01-F01|2019|wheat    |183  |39.9                  |699.1                 |73572             |2019-09-12   |2020-06-03  |
+-------+-----------+----+---------+-----+----------------------+----------------------+-

## Analytics Zone

In [10]:
from pyspark.sql import Row

marker_df = spark.createDataFrame([Row(status="initialized")])

marker_df.write.mode("overwrite").parquet(f"{CURATED}/_init")
marker_df.write.mode("overwrite").parquet(f"{ANALYTICS}/_init")

print("Curated and analytics zones initialized.")

Curated and analytics zones initialized.


## Final Check


In [11]:
print("\n=== Stage 1 Ready Check ===")

check_paths = [
    f"{LANDING}/crop-records",
    f"{LANDING}/equipment-usage",
    f"{LANDING}/market-prices",
    f"{LANDING}/soil-sensors",
    f"{LANDING}/weather-data",
    f"{CURATED}/_init",
    f"{ANALYTICS}/_init",
]

for path in check_paths:
    try:
        if path.endswith("_init"):
            df = spark.read.parquet(path)
        elif "equipment-usage" in path or "soil-sensors" in path:
            df = spark.read.json(path)
        else:
            df = spark.read.option("header", True).csv(path)

        print(f"[OK] {path} -> {df.count()} rows")

    except Exception as e:
        print(f"[ERR] {path} -> {e}")


=== Stage 1 Ready Check ===
[OK] hdfs://namenode:9000/agriflow/landing/crop-records -> 100000 rows
[OK] hdfs://namenode:9000/agriflow/landing/equipment-usage -> 80000 rows
[OK] hdfs://namenode:9000/agriflow/landing/market-prices -> 20000 rows
[OK] hdfs://namenode:9000/agriflow/landing/soil-sensors -> 500000 rows
[OK] hdfs://namenode:9000/agriflow/landing/weather-data -> 150000 rows
[OK] hdfs://namenode:9000/agriflow/curated/_init -> 1 rows
[OK] hdfs://namenode:9000/agriflow/analytics/_init -> 1 rows
